# 🗣️ Local TTS Audio Generator — Laptop Version

This notebook generates diverse synthetic speech samples directly on your **laptop** with **zero GPU overhead**.

### Engines & Voices Included:
- **Microsoft Edge-TTS:** Neural text-to-speech across American, British, and Indian accents, plus Hindi and Telugu.
- **Google Translate TTS (`gTTS`):** Alternative vocoder engine.
- **Interactive Audio Player:** Listen to generated clips directly inside the notebook.
- **Automatic Manifest Export:** Saves metadata to `data_v2/tts/tts_metadata.csv`.

--- 
## 1. Environment & Setup

In [ ]:
import asyncio
import sys
from pathlib import Path
import pandas as pd
from IPython.display import Audio, display
import edge_tts
from gtts import gTTS

# Ensure project root is accessible
PROJECT_ROOT = Path("..").resolve().parent if Path("..").resolve().name == "notebooks" else Path(".").resolve()
TTS_OUT_DIR = PROJECT_ROOT / "data_v2" / "tts"
TTS_OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"TTS Output Directory: {TTS_OUT_DIR}")

--- 
## 2. Define Multilingual Voice Profiles & Text Prompts

In [ ]:
# Multi-language phrase bank
PHRASES = [
    # English phrases
    {"text": "Your account security verification code is four eight two nine one.", "lang": "en"},
    {"text": "Artificial intelligence speech models require robust testing across accents.", "lang": "en"},
    {"text": "The quick brown fox jumps gracefully over the lazy golden retriever.", "lang": "en"},
    {"text": "We noticed an unauthorized login attempt from a new mobile device.", "lang": "en"},
    {"text": "Good morning, thank you for calling customer service, how can I assist you today?", "lang": "en"},
    # Hindi phrases
    {"text": "नमस्ते, आपका बैंक खाता सफलतापूर्वक सत्यापित कर दिया गया है।", "lang": "hi"},
    {"text": "कृत्रिम बुद्धिमत्ता आधुनिक तकनीक को तेज़ी से बदल रही है।", "lang": "hi"},
    # Telugu phrases
    {"text": "నమస్కారం, మీ ఖాతా వివరాలు విజయవంతంగా ధృవీకరించబడ్డాయి.", "lang": "te"},
    {"text": "కృత్రిమ మేధస్సు భవిష్యత్తు సాంకేతిక పరిజ్ఞానాన్ని నిర్దేశిస్తుంది.", "lang": "te"}
]

# Voice configs with split tags (holding out GuyNeural for test)
VOICE_CATALOG = [
    {"engine": "edge_tts", "voice": "en-US-JennyNeural", "speaker": "edge_Jenny", "lang": "en", "split": "train"},
    {"engine": "edge_tts", "voice": "en-GB-SoniaNeural", "speaker": "edge_Sonia", "lang": "en", "split": "train"},
    {"engine": "edge_tts", "voice": "en-IN-NeerjaNeural", "speaker": "edge_Neerja", "lang": "en", "split": "dev"},
    {"engine": "edge_tts", "voice": "hi-IN-SwaraNeural", "speaker": "edge_Swara", "lang": "hi", "split": "dev"},
    {"engine": "edge_tts", "voice": "te-IN-ShrutiNeural", "speaker": "edge_Shruti", "lang": "te", "split": "dev"},
    {"engine": "edge_tts", "voice": "en-US-GuyNeural", "speaker": "edge_Guy", "lang": "en", "split": "heldout_eval"},
    {"engine": "gtts", "voice": "google_en", "speaker": "gtts_default", "lang": "en", "split": "dev"}
]

print(f"Catalog has {len(VOICE_CATALOG)} voices across 3 languages.")

--- 
## 3. Asynchronous Audio Synthesis Engine

In [ ]:
async def generate_edge(text, voice, out_path):
    comm = edge_tts.Communicate(text, voice)
    await comm.save(str(out_path))

def generate_gtts_clip(text, out_path):
    tts = gTTS(text=text, lang="en")
    tts.save(str(out_path))

metadata_records = []

for v in VOICE_CATALOG:
    engine = v["engine"]
    voice = v["voice"]
    speaker = v["speaker"]
    lang = v["lang"]
    split = v["split"]
    
    matching = [p for p in PHRASES if p["lang"] == lang]
    print(f"Generating voice: {voice} ({len(matching)} clips)...")
    
    for idx, item in enumerate(matching):
        fn = f"{speaker}_{idx:02d}.mp3"
        out_file = TTS_OUT_DIR / fn
        
        if not out_file.exists():
            if engine == "edge_tts":
                await generate_edge(item["text"], voice, out_file)
            else:
                generate_gtts_clip(item["text"], out_file)
        
        metadata_records.append({
            "path": str(out_file).replace("\\", "/"),
            "label": "spoof",
            "source": engine,
            "speaker": speaker,
            "generator": voice,
            "text": item["text"],
            "language": lang,
            "split": split
        })

df_meta = pd.DataFrame(metadata_records)
meta_path = TTS_OUT_DIR / "tts_metadata.csv"
df_meta.to_csv(meta_path, index=False)
print(f"\nSuccessfully synthesized {len(df_meta)} clips. Metadata saved to {meta_path}")

--- 
## 4. In-Notebook Audio Player
Listen to any synthesized sample directly inside Jupyter:

In [ ]:
# Select a sample to play
sample_files = list(TTS_OUT_DIR.glob("*.mp3"))
if sample_files:
    print(f"Playing audio: {sample_files[0].name}")
    display(Audio(str(sample_files[0])))
    
    if len(sample_files) > 5:
        print(f"Playing audio: {sample_files[5].name}")
        display(Audio(str(sample_files[5])))
else:
    print("No audio files found. Run the generation cell above first.")